<a href="https://colab.research.google.com/github/boss-defender/Smart-Fine-Tune/blob/main/SmartFineTuner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ⬇🚀 1. Universal Smart Fine Tuning (One-Click Auto Fine-Tuner)
# ==============================================================================
# ⚡ UNIVERSAL MODEL & DATASET AUTO-TRAINER
# Fine-tune ANY Hugging Face model on ANY dataset with full automation.
# ==============================================================================

# --- 🎛️ CONFIGURATION & FORM INPUTS ---
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"           #@param {type:"string"}
DATASET_NAME = "tatsu-lab/alpaca"                  #@param {type:"string"}
DATASET_CONFIG = ""                                #@param {type:"string"}
MAX_SAMPLES = ""                                  #@param {type:"string"}
MAX_SEQ_LENGTH = ""                               #@param {type:"string"}
LOAD_IN_4BIT = True                                #@param {type:"boolean"}
HF_TOKEN = ""                                      #@param {type:"string"}

# Hyperparameters (Leave empty or set to 0 for automatic calculation)
LEARNING_RATE = ""                                 #@param {type:"string"}
MAX_STEPS = ""                                     #@param {type:"string"}
SAVE_STEPS = ""                                    #@param {type:"string"}

import os
import sys
import hashlib
import json
import torch

# --- STEP 1: HARDWARE DETECTION & AUTO-TUNING ---
print("🔍 [1/8] Verifying Hardware & Auto-Tuning Parameters...")
if not torch.cuda.is_available():
    raise SystemError("❌ No GPU found! Please change Colab Runtime to T4 GPU (Runtime > Change runtime type > T4 GPU).")

gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"✅ GPU Detected: {gpu_name} ({total_vram:.2f} GB VRAM)")

# Auto-calculate micro-batch size and gradient accumulation based on VRAM
if total_vram < 12:
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
elif total_vram < 20:
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
else:
    BATCH_SIZE = 4
    GRAD_ACCUM = 2

print(f"⚡ Hardware Auto-Tuned: Micro-Batch Size = {BATCH_SIZE}, Grad Accumulation = {GRAD_ACCUM}\n")

# --- STEP 2: MOUNT GOOGLE DRIVE ---
print("📂 [2/8] Connecting Google Drive for Checkpoints...")
DRIVE_BASE_DIR = "/content/drive/MyDrive/unsloth_checkpoints"
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("✅ Google Drive connected successfully!")
except Exception as e:
    print(f"⚠️ Google Drive not mounted ({e}). Using local storage /content/checkpoints")
    DRIVE_BASE_DIR = "/content/checkpoints"

# --- STEP 3: INSTALL DEPENDENCIES (CLEAN WITHOUT BROKEN PINS) ---
print("\n🔄 [3/8] Installing Unsloth, Transformers & Core Dependencies...")
!pip install --quiet --upgrade pip setuptools wheel
!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" unsloth_zoo
!pip install --quiet trl peft accelerate bitsandbytes datasets huggingface_hub hf_transfer transformers
print("✅ Packages installed successfully!\n")

# --- STEP 4: AUTO PARAMETER RESOLVER ---
# Auto Learning Rate
try:
    lr_val = float(LEARNING_RATE) if LEARNING_RATE is not None and str(LEARNING_RATE).strip() != "" else 2e-4
    if lr_val <= 0: lr_val = 2e-4
except Exception:
    lr_val = 2e-4
LEARNING_RATE_RESOLVED = lr_val

# Auto Max Seq Length
try:
    seq_len_val = int(MAX_SEQ_LENGTH) if MAX_SEQ_LENGTH is not None and str(MAX_SEQ_LENGTH).strip() != "" else 2048
    if seq_len_val <= 0: seq_len_val = 2048
except Exception:
    seq_len_val = 2048
MAX_SEQ_LENGTH_RESOLVED = seq_len_val

print(f"⚙️ Auto Parameters Resolved: Learning Rate = {LEARNING_RATE_RESOLVED}, Max Seq Length = {MAX_SEQ_LENGTH_RESOLVED}")

# --- STEP 5: EXPERIMENT ISOLATION & HASH LOCK ---
config_str = f"{MODEL_NAME}_{DATASET_NAME}_{DATASET_CONFIG}_{LEARNING_RATE_RESOLVED}_{BATCH_SIZE}_{GRAD_ACCUM}"
config_hash = hashlib.sha256(config_str.encode()).hexdigest()[:8]
folder_name = f"{MODEL_NAME.split('/')[-1]}__{DATASET_NAME.split('/')[-1]}__{config_hash}".replace(".", "_").replace("-", "_")
RUN_DIR = os.path.join(DRIVE_BASE_DIR, folder_name)
os.makedirs(RUN_DIR, exist_ok=True)

print(f"📂 RUN DIRECTORY: {RUN_DIR}")
print(f"🔒 Experiment SHA-256 Hash: {config_hash}\n")

# Save experiment metadata
config_file = os.path.join(RUN_DIR, "run_config.json")
run_metadata = {
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "dataset_config": DATASET_CONFIG,
    "learning_rate": LEARNING_RATE_RESOLVED,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "hash": config_hash
}
with open(config_file, "w") as f:
    json.dump(run_metadata, f, indent=2)

# --- STEP 6: DUAL ENGINE MODEL LOADER (UNSLOTH + PEFT FALLBACK) ---
print(f"📥 [4/8] Loading Base Model: '{MODEL_NAME}'...")
use_unsloth = False
model = None
tokenizer = None

try:
    print("🚀 Attempting loading with Unsloth Engine...")
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH_RESOLVED,
        dtype = None,
        load_in_4bit = LOAD_IN_4BIT,
        token = HF_TOKEN if HF_TOKEN else None,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
    use_unsloth = True
    print("✅ Unsloth Engine initialized successfully!\n")
except Exception as e:
    print(f"⚠️ Unsloth fast engine skipped or incompatible ({e}).")
    print("🔄 Seamlessly falling back to Hugging Face Transformers + PEFT Engine...")
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    
    quant_config = None
    if LOAD_IN_4BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
            bnb_4bit_use_double_quant=True
        )
    
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, 
        token=HF_TOKEN if HF_TOKEN else None,
        trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quant_config,
        device_map="auto",
        token=HF_TOKEN if HF_TOKEN else None,
        trust_remote_code=True
    )
    model = prepare_model_for_kbit_training(model)
    peft_config = LoraConfig(
        r=16,
        lora_alpha=16,
        target_modules="all-linear",
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, peft_config)
    print("✅ Hugging Face Transformers + PEFT initialized successfully!\n")

# --- STEP 7: BULLETPROOF DATASET LOADING & REPOSITORY FALLBACK ENGINE ---
print(f"📊 [5/8] Auto-Detecting & Loading Dataset: '{DATASET_NAME}'...")
from datasets import load_dataset, get_dataset_config_names
from huggingface_hub import list_repo_files

def bulletproof_load_dataset(dataset_name, dataset_config=None, token=None):
    cfg = str(dataset_config).strip() if dataset_config is not None else ""
    if cfg.lower() in ["", "auto", "none", "default", "null"]:
        cfg = None

    # Step 1: Try direct specified config
    if cfg:
        try:
            print(f"🔄 Attempting direct load with config '{cfg}'...")
            return load_dataset(dataset_name, name=cfg, token=token if token else None, trust_remote_code=True, verification_mode="no_checks")
        except Exception as e:
            print(f"⚠️ Direct load for config '{cfg}' failed ({e}). Auto-resolving configs...")

    # Step 2: Detect available configs
    configs_to_try = []
    try:
        detected_configs = get_dataset_config_names(dataset_name, token=token if token else None)
        if detected_configs:
            print(f"ℹ️ Available dataset configs: {detected_configs}")
            configs_to_try = detected_configs
    except Exception:
        pass

    if not configs_to_try:
        configs_to_try = [None]

    # Step 3: Loop configs with verification bypassed
    last_error = None
    for c in configs_to_try:
        try:
            print(f"🔄 Trying config '{c}'...")
            if c:
                return load_dataset(dataset_name, name=c, token=token if token else None, trust_remote_code=True, verification_mode="no_checks")
            else:
                return load_dataset(dataset_name, token=token if token else None, trust_remote_code=True, verification_mode="no_checks")
        except Exception as e:
            last_error = e
            print(f"⚠️ Config '{c}' load failed: {e}")

    # Step 4: Repository File Inspection Fallback (Parquet, JSON, JSONL, CSV)
    print("🚀 Native HF script builder failed. Triggering Repository File Fallback...")
    try:
        repo_files = list_repo_files(dataset_name, repo_type="dataset", token=token if token else None)
        # Try parquet
        parquet_files = [f for f in repo_files if f.endswith(".parquet")]
        if parquet_files:
            print(f"📦 Found {len(parquet_files)} Parquet files. Loading directly...")
            return load_dataset("parquet", data_files={"train": [f"hf://datasets/{dataset_name}/{pf}" for pf in parquet_files]})
        # Try json/jsonl/jsonl.zst
        json_files = [f for f in repo_files if f.endswith(".json") or f.endswith(".jsonl") or f.endswith(".jsonl.zst")]
        if json_files:
            print(f"📦 Found {len(json_files)} JSON/JSONL files. Loading directly...")
            return load_dataset("json", data_files={"train": [f"hf://datasets/{dataset_name}/{jf}" for jf in json_files]})
        # Try csv
        csv_files = [f for f in repo_files if f.endswith(".csv") or f.endswith(".tsv")]
        if csv_files:
            print(f"📦 Found {len(csv_files)} CSV files. Loading directly...")
            return load_dataset("csv", data_files={"train": [f"hf://datasets/{dataset_name}/{cf}" for cf in csv_files]})
    except Exception as e_fb:
        print(f"⚠️ Repository file fallback notice: {e_fb}")

    raise RuntimeError(f"❌ Failed to load dataset '{dataset_name}': {last_error}")

raw_ds = bulletproof_load_dataset(DATASET_NAME, DATASET_CONFIG, HF_TOKEN)

# Split Auto-Detection
split_name = None
if isinstance(raw_ds, dict) or hasattr(raw_ds, "keys"):
    available_splits = list(raw_ds.keys())
    print(f"ℹ️ Available dataset splits: {available_splits}")
    for preferred in ["train", "train_sft", "training", "train_eval", "default"]:
        if preferred in available_splits:
            split_name = preferred
            break
    if not split_name:
        split_name = available_splits[0]
    dataset = raw_ds[split_name]
else:
    dataset = raw_ds

print(f"✅ Loaded split '{split_name}' with {len(dataset)} total samples.")

# Robust Sample Slicing (Train on full dataset if MAX_SAMPLES is 0, negative, or blank)
try:
    max_samples_int = int(MAX_SAMPLES) if MAX_SAMPLES is not None and str(MAX_SAMPLES).strip() != "" else 0
except Exception:
    max_samples_int = 0

if max_samples_int > 0 and max_samples_int < len(dataset):
    dataset = dataset.select(range(max_samples_int))
    print(f"✂️ Sliced dataset to MAX_SAMPLES = {len(dataset)}")
else:
    print(f"ℹ️ Training on FULL dataset ({len(dataset)} total samples).")

# Universal Row Parser & Formatter
def universal_format_row(row):
    # 1. Check for OpenAI 'messages' format
    if "messages" in row and isinstance(row["messages"], list):
        text = ""
        for msg in row["messages"]:
            if isinstance(msg, dict):
                role = msg.get("role", "user")
                content = msg.get("content", "")
                text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        if text.strip():
            return {"text": text.strip()}

    # 2. Check for ShareGPT / Bespoke 'conversations' format
    if "conversations" in row and isinstance(row["conversations"], list):
        text = ""
        for msg in row["conversations"]:
            if isinstance(msg, dict):
                role = msg.get("from", msg.get("role", "user"))
                content = msg.get("value", msg.get("content", ""))
                if role in ["human", "user"]:
                    role = "user"
                elif role in ["gpt", "assistant", "bot"]:
                    role = "assistant"
                text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        if text.strip():
            return {"text": text.strip()}

    # 3. Check for Instruction / Response pairs
    inst_keys = ["instruction", "prompt", "question", "query", "problem", "input_text"]
    out_keys = ["output", "response", "answer", "solution", "completion", "target", "output_text"]
    inp_keys = ["input", "context"]

    found_inst = next((k for k in inst_keys if k in row and row[k]), None)
    found_out = next((k for k in out_keys if k in row and row[k]), None)
    found_inp = next((k for k in inp_keys if k in row and row[k]), None)

    if found_inst and found_out:
        inst_val = str(row[found_inst]).strip()
        out_val = str(row[found_out]).strip()
        inp_val = str(row[found_inp]).strip() if found_inp else ""
        full_user = f"{inst_val}\nContext: {inp_val}" if inp_val else inst_val
        formatted = f"<|im_start|>user\n{full_user}<|im_end|>\n<|im_start|>assistant\n{out_val}<|im_end|>"
        return {"text": formatted}

    # 4. Check for Pair / Classification style
    if "premise" in row and "hypothesis" in row:
        lbl = f"\nLabel: {row['label']}" if "label" in row else ""
        return {"text": f"<|im_start|>user\nPremise: {row['premise']}\nHypothesis: {row['hypothesis']}{lbl}<|im_end|>"}
    if "sentence1" in row and "sentence2" in row:
        lbl = f"\nLabel: {row['label']}" if "label" in row else ""
        return {"text": f"<|im_start|>user\nSentence 1: {row['sentence1']}\nSentence 2: {row['sentence2']}{lbl}<|im_end|>"}

    # 5. Check for Single text column
    text_keys = ["text", "content", "body", "document", "raw", "sentence", "prompt", "inputs", "code"]
    found_text = next((k for k in text_keys if k in row and row[k]), None)
    if found_text:
        return {"text": str(row[found_text]).strip()}

    # 6. Generic Fallback: Combine all non-empty scalar fields
    parts = []
    for k, v in row.items():
        if v is not None and isinstance(v, (str, int, float, bool)):
            v_str = str(v).strip()
            if v_str:
                parts.append(f"{k}: {v_str}")
    if parts:
        return {"text": "\n".join(parts)}
    
    return {"text": ""}

print("🪄 Processing and formatting dataset schema into standardized text field...")
formatted_dataset = dataset.map(universal_format_row, batched=False)
formatted_dataset = formatted_dataset.filter(lambda x: len(x.get("text", "").strip()) > 0)
print(f"✅ Dataset successfully formatted! {len(formatted_dataset)} valid rows ready for training.\n")

# --- STEP 8: AUTO RESOLVE MAX_STEPS & SAVE_STEPS ---
try:
    user_max_steps = int(MAX_STEPS) if MAX_STEPS is not None and str(MAX_STEPS).strip() != "" else 0
except Exception:
    user_max_steps = 0

if user_max_steps > 0:
    MAX_STEPS_RESOLVED = user_max_steps
else:
    # Train for ~60 steps or 1 full epoch, whichever is reasonable
    total_ds_size = len(formatted_dataset)
    steps_per_epoch = max(1, total_ds_size // (BATCH_SIZE * GRAD_ACCUM))
    MAX_STEPS_RESOLVED = min(100, max(30, steps_per_epoch))

try:
    user_save_steps = int(SAVE_STEPS) if SAVE_STEPS is not None and str(SAVE_STEPS).strip() != "" else 0
except Exception:
    user_save_steps = 0

if user_save_steps > 0:
    SAVE_STEPS_RESOLVED = user_save_steps
else:
    SAVE_STEPS_RESOLVED = max(10, MAX_STEPS_RESOLVED // 3)

print(f"⚙️ Training Steps Auto-Tuned: MAX_STEPS = {MAX_STEPS_RESOLVED}, SAVE_STEPS = {SAVE_STEPS_RESOLVED}")

# --- STEP 9: SETUP TRAINER & AUTO-RESUME CHECKPOINTS ---
print("🛠️ [6/8] Initializing SFTTrainer & Auto-Resume Checkpoints...")
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Check Google Drive for existing checkpoints to auto-resume
checkpoints = [d for d in os.listdir(RUN_DIR) if d.startswith("checkpoint-")]
resume_from_checkpoint = None
if checkpoints:
    latest_checkpoint = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
    resume_from_checkpoint = os.path.join(RUN_DIR, latest_checkpoint)
    print(f"🔄 AUTO-RESUMING from existing checkpoint: {latest_checkpoint}")
else:
    print("🚀 STARTING FRESH TRAINING RUN...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH_RESOLVED,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps = 5,
        max_steps = MAX_STEPS_RESOLVED,
        learning_rate = LEARNING_RATE_RESOLVED,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = RUN_DIR,
        save_strategy = "steps",
        save_steps = SAVE_STEPS_RESOLVED,
        save_total_limit = 3,
    ),
)

# --- STEP 10: LAUNCH TRAINING ---
print("🔥 [7/8] Training in progress...")
trainer.train(resume_from_checkpoint=resume_from_checkpoint)
print("🎉 Training Completed Successfully!\n")

# --- STEP 11: EXPORT MERGED BASE MODEL ---
EXPORT_DIR = "/content/merged_16bit_model"
print(f"📦 [8/8] Exporting 16-bit merged base model to: '{EXPORT_DIR}'...")
if use_unsloth:
    model.save_pretrained_merged(EXPORT_DIR, tokenizer, save_method = "merged_16bit")
else:
    model.save_pretrained(EXPORT_DIR)
    tokenizer.save_pretrained(EXPORT_DIR)
print("✅ Fine-tuned model exported successfully!")

**⚡ Optional: Upload Merged Model to Hugging Face Hub**

In [ ]:
# @title ⬇📤 2. Upload to Hugging Face Hub
# ==============================================================================
# LIGHTWEIGHT HF UPLOADER (NO UNSLOTH NEEDED! 🚀)
# ==============================================================================
!pip install --quiet huggingface_hub

from huggingface_hub import HfApi
import os

# 1. CONFIGURATION PANEL 🎛️
HF_WRITE_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}
HF_REPO_NAME = "your-username/my-finetuned-model"        #@param {type:"string"}
FOLDER_PATH = "/content/merged_16bit_model"             #@param {type:"string"}
REPO_VISIBILITY = "public"                             #@param ["private", "public"]

# 2. VERIFY FOLDER EXISTS
if not os.path.exists(FOLDER_PATH):
    raise FileNotFoundError(f"❌ Cannot find folder at: {FOLDER_PATH}")

# 3. INITIALIZE HF API & CREATE REPO
api = HfApi(token=HF_WRITE_TOKEN)

print(f"📁 Preparing Hugging Face repository '{HF_REPO_NAME}' ({REPO_VISIBILITY.upper()})...")
api.create_repo(
    repo_id=HF_REPO_NAME,
    private=(REPO_VISIBILITY == "private"),
    exist_ok=True,
    repo_type="model"
)

# 4. PUSH FOLDER TO HUGGING FACE
print(f"🚀 Uploading all files from '{FOLDER_PATH}'...")
api.upload_folder(
    folder_path=FOLDER_PATH,
    repo_id=HF_REPO_NAME,
    repo_type="model",
)

print(f"\n🎉 BOOM! Your model is live at: https://huggingface.co/{HF_REPO_NAME}")

#🛠️ Install hugging face tool

*   **pip install huggingface_hub**

or,
*   **pip install huggingface_hub --break-system-packages**


#📝 Command prompt

**</>**  **hf download your-username/my-finetuned-model  --local-dir /path/to/dir/[Folder where you will save files]/**

In [ ]:
# @title ⬇️ 3. Auto-Download Model (Browser Zip Download)
# ==============================================================================
import os
from google.colab import files

model_folder = "/content/merged_16bit_model" #@param {type:"string"}

if os.path.exists(model_folder):
    print("📦 Compressing model folder...")
    !zip -r /content/model.zip {model_folder}
    print("⬇️ Triggering browser download now...")
    files.download("/content/model.zip")
else:
    print(f"❌ Folder missing at {model_folder}! Please verify training export.")

---
**🪄 Single Line Command Prompt to make it ready to chat in Ollama or LM Studio**
---

🎯 **python convert_hf_to_gguf.py "/path/to/dir/[Folder Name Where All Files Saved]" --outfile "/path/to/dir/[AI Model Name].gguf" --outtype auto**

or,

🎯 **./venv/bin/python convert_hf_to_gguf.py "/path/to/dir/[Folder Name Where All Files Saved]" --outfile "/path/to/dir/[AI Model Name].gguf" --outtype auto**

---

**👇 If you want Q4_K_M or anyother format**
---

🎯 **python convert_hf_to_gguf.py "/path/to/dir/[Folder Name Where All Files Saved]" --outfile "/path/to/dir/[Temp Name].gguf" --outtype f16 && ./llama-quantize "/path/to/dir/[Temp Name].gguf" "/path/to/dir/[Final Name].gguf" Q4_K_M && rm "/path/to/dir/[Temp Name].gguf"**

or,

🎯 **./venv/bin/python convert_hf_to_gguf.py "/path/to/dir/[Folder Name Where All Files Saved]" --outfile "/path/to/dir/[Temp Name].gguf" --outtype f16 && ./build/bin/llama-quantize "/path/to/dir/[Temp Name].gguf" "/path/to/dir/[Final Name].gguf" Q4_K_M && rm "/path/to/dir/[Temp Name].gguf"**

---